In [63]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [64]:
data_dir = "data/data.txt"
text = open(data_dir, 'r').read() # load all the data as simple string

# Get all unique characters in the text as vocabulary
chars = list(set(text))
vocab_size = len(chars)
vocab_size

89

In [65]:
# build the character level tokenizer
chr_to_idx = {c:i for i, c in enumerate(chars)}
idx_to_chr = {i:c for i, c in enumerate(chars)}

def encode(input_text: str) -> list[int]:
    return [chr_to_idx[t] for t in input_text]

def decode(input_tokens: list[int]) -> str:
    return "".join([idx_to_chr[i] for i in input_tokens])

In [66]:
print (encode ("Hello world") )
print (decode (encode ("Hello world") ))

[74, 14, 53, 53, 38, 44, 47, 38, 54, 53, 66]
Hello world


In [67]:
# use cpu or gpu based on your system
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"

# convert our text data into tokenized tensor
data = torch.tensor(encode(text), dtype=torch.long, device=device)

In [68]:
train_batch_size = 16  # training batch size
eval_batch_size = 8  # evaluation batch size
context_length = 256  # number of tokens processed in a single batch
train_split = 0.8  # percentage of data to use from total data for training

# split data into train and eval
n_data = len(data)
train_data = data[:int(n_data * train_split)]
eval_data = data[int(n_data * train_split):]


class DataLoader:
    def __init__(self, tokens, batch_size, context_length) -> None:
        self.tokens = tokens
        self.batch_size = batch_size
        self.context_length = context_length
        self.current_position = 0

    def get_batch(self) -> torch.tensor:
        b, c = self.batch_size, self.context_length
        
        # Safety Check: If the cursor went past the end, loop it back to start
        if self.current_position >= len(self.tokens):
            self.current_position = self.current_position % len(self.tokens)
        
        start_pos = self.current_position
        end_pos = self.current_position + b * c + 1 
        
        add_data = -1
        
        # Handle wrapping if we run off the end of the text
        if end_pos > len(self.tokens):
            add_data = end_pos - len(self.tokens)
            end_pos = len(self.tokens)
            
        d = self.tokens[start_pos:end_pos]
        
        if add_data != -1:
            d = torch.cat([d, self.tokens[:add_data]])
            
        x = (d[:-1]).view(b, c)  # inputs
        y = (d[1:]).view(b, c)   # targets
        
        # Update position for next batch
        self.current_position += b * c 
        
        return x, y

# IMPORTANT: You must re-run these lines to apply the fix
train_loader = DataLoader(train_data, train_batch_size, context_length)
eval_loader = DataLoader(eval_data, eval_batch_size, context_length)

In [69]:
# used to define size of embeddings
d_model = vocab_size 
d_model


89

In [70]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model) # word token embeddings

    def forward(self, inputs, targets = None):
        logits = self.wte(inputs) # dim -> batch_size, sequence_length, d_model
        loss = None
        if targets != None:
            batch_size, sequence_length, d_model = logits.shape
            # to calculate loss for all token embeddings in a batch
            # kind of a requirement for cross_entropy
            logits = logits.view(batch_size * sequence_length, d_model)
            targets = targets.view(batch_size * sequence_length)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, inputs, max_new_tokens):
        # this will store the model outputs along with the initial input sequence
        # make a copy so that it doesn't interfare with model 
        for _ in range(max_new_tokens):
            # we only pass targets on training to calculate loss
            logits, _ = self(inputs)  
            # for all the batches, get the embeds for last predicted sequence
            logits = logits[:, -1, :] 
            probs = F.softmax(logits, dim=1)            
            # get the probable token based on the input probs
            idx_next = torch.multinomial(probs, num_samples=1) 

            inputs = torch.cat([inputs, idx_next], dim=1)
        # as the inputs has all model outputs + initial inputs, we can use it as final output
        return inputs

m = GPT(vocab_size=vocab_size, d_model=d_model).to(device)

In [71]:
lr = 1e-3
optim = torch.optim.AdamW(m.parameters(), lr=lr)

In [72]:
epochs = 20000
eval_steps = 1000 # perform evaluation in every n steps
for ep in range(epochs):
    xb, yb = train_loader.get_batch()

    logits, loss = m(xb, yb)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    optim.step()

    if ep % eval_steps == 0 or ep == epochs-1:
        m.eval()
        with torch.no_grad():
            xvb, yvb = eval_loader.get_batch()
            _, e_loss = m(xvb, yvb)

            print(f"Epoch: {ep}\tlr: {lr}\ttrain_loss: {loss}\teval_loss: {e_loss}")
        m.train() # back to training mode

Epoch: 0	lr: 0.001	train_loss: 4.916573524475098	eval_loss: 4.947132587432861
Epoch: 1000	lr: 0.001	train_loss: 3.7416133880615234	eval_loss: 3.6879875659942627
Epoch: 2000	lr: 0.001	train_loss: 2.9959628582000732	eval_loss: 3.0585482120513916
Epoch: 3000	lr: 0.001	train_loss: 2.6943228244781494	eval_loss: 2.78721284866333
Epoch: 4000	lr: 0.001	train_loss: 2.5289812088012695	eval_loss: 2.492760181427002
Epoch: 5000	lr: 0.001	train_loss: 2.466856002807617	eval_loss: 2.4524576663970947
Epoch: 6000	lr: 0.001	train_loss: 2.4809207916259766	eval_loss: 2.335808515548706
Epoch: 7000	lr: 0.001	train_loss: 2.436586380004883	eval_loss: 2.504664182662964
Epoch: 8000	lr: 0.001	train_loss: 2.355449914932251	eval_loss: 2.4923431873321533
Epoch: 9000	lr: 0.001	train_loss: 2.380387306213379	eval_loss: 2.3716397285461426
Epoch: 10000	lr: 0.001	train_loss: 2.3112597465515137	eval_loss: 2.2899796962738037
Epoch: 11000	lr: 0.001	train_loss: 2.371169090270996	eval_loss: 2.4309115409851074
Epoch: 12000	lr: 

In [73]:
# 1. Choose your starting text
start_str = "Love"

# 2. Encode it to integers and move to GPU/CPU
# encode() gives a list: [23, 5, 8]
# tensor(...) turns it into a tensor
# .unsqueeze(0) changes shape from [3] to [1, 3] (Batch size of 1)
input_tensor = torch.tensor(encode(start_str), dtype=torch.long, device=device).unsqueeze(0)

# 3. Tell the model to generate 100 new tokens
# m is the name of your model variable from previous steps
print("Generating...")
output_tensor = m.generate(input_tensor, max_new_tokens=1000)

# 4. Decode the result back to text
# output_tensor[0] selects the first (and only) batch
# .tolist() converts it back to a standard Python list
generated_text = decode(output_tensor[0].tolist())

print("--- Result ---")
print(generated_text)

Generating...
--- Result ---
Loved
Cahofoucok, o younouphivit blart ou, mye ry, forndseves cthalythtighetyomevigr s ye lllearof gheds a myon tseer tss
l guratheve c om w yo oferf m),



Whe rur y th lofot lyt, ht yomy s f
And au ta l atory h morere

Ungrofo g
Yomorienom unthe nd
And e),
I g I wn l migur aivom fatrgimelin wa he
Ohon,
Bar srowinof llo st'The medo id'lse uy,
Thind whe tove do,
Amecaspad ld pr wabat,

Do tu t'damyous y, whe st youe cou jatin,
Lifirs rtak f br owheare sp ntild s,
Thoure halomy t)
Sherilatachous raured yoneps hay hin'Cowouco ou yng t'tothighe we l caing topat myo brve ckecke I tr,
Whed en'mith cr Myou"Fout yo,

An ikes t am geext kn'tod thet w yous sckiemep iert that m igoucof a ow hisli nouse s
Sthty isthowixind s wselataledomevere cetalous yb I towovid inke in'ss
Ban

co Is,
I tove
Yok nd by h)


I jupindad ane w d midou m s towabed tig tind ovis ais fond the l tt ofun ande w by?)
I'r t akiereane
Folou ombymakn baithel?
I'l qug,
I o,

Alywed d wa y s,
Angh

Part 2

In [83]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class PositionalEncoding(nn.Module):
    def __init__(self, context_length, d_model) -> None:
        super().__init__()
        # 1. Create a blank matrix of shape (context_length, d_model)
        pe = torch.zeros(context_length, d_model)
        
        # 2. Create a vector with positions [0, 1, 2, ..., context_length-1]
        position = torch.arange(0, context_length, dtype=torch.float).unsqueeze(1)
        
        # 3. Create the divisor terms (frequencies)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # 4. Compute Sine and Cosine patterns
        pe[:, 0::2] = torch.sin(position * div_term)
        
        # FIX: Handle odd d_model sizes (like 89) by slicing the cosine result
        cos_part = torch.cos(position * div_term)
        pe[:, 1::2] = cos_part[:, :pe[:, 1::2].size(1)]
        
        pe = pe.unsqueeze(0)  # Shape: (1, context_length, d_model)
        
        # Register as a buffer (not a trainable parameter)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Add the positional encodings to the input embeddings
        return x + self.pe[:, :x.size(1), :]

In [84]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model) # Word Token Embeddings
        
        # 1. NEW: Initialize Positional Encodings
        self.wpe = PositionalEncoding(context_length, d_model)
        
        # 2. NEW: Fully Connected Network (The "Expansion Chamber")
        self.fcn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), # Expand to 4x
            nn.GELU(),                       # Activation
            nn.Linear(4 * d_model, d_model)  # Compress back
        )

    def forward(self, inputs, targets=None):
        # A. Embed the inputs
        logits = self.wte(inputs)
        
        # B. Add Positional Information
        logits = self.wpe(logits)
        
        # C. Pass through the Expansion Chamber
        logits = self.fcn(logits)
        
        loss = None
        if targets is not None:
            batch_size, sequence_length, d_model = logits.shape
            logits = logits.view(batch_size * sequence_length, d_model)
            targets = targets.view(batch_size * sequence_length)
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss

    def generate(self, inputs, max_new_tokens):
        # 3. NEW: Updated Generation Logic
        # Create a separate buffer for the full output
        output = inputs.clone()
        
        for _ in range(max_new_tokens):
            current_seq_length = inputs.size(1)
            
            # TRUNCATION: If input is too long for the Position Map, chop off the beginning
            if current_seq_length > context_length:
                inputs = inputs[:, -context_length:]
            
            # Get predictions
            logits, _ = self(inputs)
            
            # Focus on the last token
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            
            # Sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            
            # Append to inputs (for next step) and output (for final result)
            inputs = torch.cat([inputs, idx_next], dim=1)
            output = torch.cat([output, idx_next], dim=1)
            
        return [decode(out.tolist()) for out in output]

# Re-initialize the model with these updates
m = GPT(vocab_size, d_model).to(device)
print("Model updated with Positional Encoding & FCN.")

Model updated with Positional Encoding & FCN.


In [ ]:
# Setup Optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
epochs = 2000

print(f"Starting training for {epochs} epochs...")

for ep in range(epochs):
    # 1. Get a batch of data
    xb, yb = train_loader.get_batch()
    
    # 2. Forward pass (Calculate predictions and loss)
    logits, loss = m(xb, yb)
    
    # 3. Backward pass (Update weights)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    # Print progress every 500 epochs
    if ep % 500 == 0 or ep == epochs - 1:
        print(f"Epoch: {ep}\tLoss: {loss.item():.4f}")

print("Training Complete.")

# --- Test Generation ---
print("\n--- Generating Text ---")
start_str = "The "
input_tensor = torch.tensor(encode(start_str), dtype=torch.long, device=device).unsqueeze(0)
generated_text = m.generate(input_tensor, max_new_
tokens=100)[0]
print(generated_text)


Starting training for 2000 epochs...
Epoch: 0	Loss: 2.2671
Epoch: 500	Loss: 2.4298
Epoch: 1000	Loss: 2.2577
Epoch: 1500	Loss: 2.3257
Epoch: 1999	Loss: 2.3024
Training Complete.

--- Generating Text ---
The yow,Wou014ictht I'Thero t kest
Ohinas we
An kn cerll,tike I whed ho yow
Canyoure likir me wre wa yo 


In [87]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SelfAttention(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        
        # The Three Beams: Query, Key, Value
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        
        # Final aggregation projector
        self.fc_out = nn.Linear(d_model, d_model)
        
        # Randomly disable 20% of connections to prevent over-reliance (Dropout)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, inputs: torch.Tensor):
        B, seq_length, d_model = inputs.shape
        
        # 1. Project inputs into Q, K, and V
        Q = self.query(inputs)
        K = self.key(inputs)
        V = self.value(inputs)
        
        # 2. Compute Attention Scores (The Web Strength)
        # We check how much Q aligns with K (Matrix Multiplication)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1))
        
        # 3. Apply The Time Shield (Masking)
        # Create a matrix of 1s and 0s (upper triangle is 1)
        mask = torch.triu(torch.ones(seq_length, seq_length), diagonal=1).bool().to(inputs.device)
        # Fill future positions with negative infinity (so Softmax turns them to 0)
        attention_scores = attention_scores.masked_fill(mask, float('-inf'))
        
        # 4. Normalize to probabilities (0.0 to 1.0)
        attention_weights = torch.softmax(attention_scores, dim=-1)
        
        # 5. Aggregate Information (Weighted Sum of Values)
        attention_output = torch.matmul(attention_weights, V)

        # 6. Final Linear Transformation
        out = self.fc_out(attention_output)
        
        return out

print("Self-Attention Mechanism Online.")

Self-Attention Mechanism Online.


In [88]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model)
        self.wpe = PositionalEncoding(context_length, d_model) # Defined in previous steps
        
        # The Brain: Self Attention
        self.att = SelfAttention(d_model)
        
        # The Stabilizers: Layer Norms
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        # The Expansion Chamber: Feed Forward Network
        self.fcn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        
        self.dropout = nn.Dropout(0.2)

    def forward(self, inputs, targets=None):
        # 1. Embeddings & Position
        logits = self.wte(inputs)
        logits = self.wpe(logits)
        
        # 2. Block 1: Attention + Residual + Norm
        # We calculate Attention
        att_logits = self.att(logits)
        # We ADD it to original logits (Residual) and Normalize
        adn_logits = self.ln1(logits + att_logits)
        
        # 3. Dropout (Randomly cut connections)
        logits = self.dropout(adn_logits)
        
        # 4. Block 2: FCN + Residual + Norm
        # We calculate FCN
        fcn_logits = self.fcn(logits)
        # We ADD it to the previous step (Residual) and Normalize
        logits = self.ln2(logits + fcn_logits) # Note: 'logits' here implies the flow, not just raw scores
        
        loss = None
        if targets is not None:
            batch_size, sequence_length, d_model = logits.shape
            logits = logits.view(batch_size * sequence_length, d_model)
            targets = targets.view(batch_size * sequence_length)
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss

    # (Generate function remains the same as previous)
    def generate(self, inputs, max_new_tokens):
        output = inputs.clone()
        for _ in range(max_new_tokens):
            current_seq_length = inputs.size(1)
            if current_seq_length > context_length:
                inputs = inputs[:, -context_length:]
            logits, _ = self(inputs)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat([inputs, idx_next], dim=1)
            output = torch.cat([output, idx_next], dim=1)
        return [decode(out.tolist()) for out in output]

# Update the model on the GPU
m = GPT(vocab_size, d_model).to(device)
print("GPT Updated with Attention, Normalization, and Residuals.")

GPT Updated with Attention, Normalization, and Residuals.


In [89]:
# --- Resolution Upgrade ---
d_model = 512  # Upgrading from 89 to 512 dimensions

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        # 1. Embeddings now project to 512 dimensions
        self.wte = nn.Embedding(vocab_size, d_model)
        self.wpe = PositionalEncoding(context_length, d_model)
        
        self.att = SelfAttention(d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        self.fcn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.dropout = nn.Dropout(0.2)
        
        # NEW: The Final Lens (Project 512 back to vocab_size)
        self.linear1 = nn.Linear(d_model, vocab_size)

    def forward(self, inputs, targets=None):
        logits = self.wte(inputs)
        logits = self.wpe(logits)
        
        att_logits = self.att(logits)
        adn_logits = self.ln1(logits + att_logits)
        
        logits = self.dropout(adn_logits)
        
        # Important: pass the 'adn_logits' (the result of the first block) into the skip connection
        fcn_logits = self.fcn(logits)
        logits = self.ln2(logits + fcn_logits) 
        
        # NEW: Apply the final lens
        logits = self.linear1(logits)
        
        loss = None
        if targets is not None:
            batch_size, sequence_length, output_dim = logits.shape
            logits = logits.view(batch_size * sequence_length, output_dim)
            targets = targets.view(batch_size * sequence_length)
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss
    
    # (Generate function needs to be included in the class definition)
    def generate(self, inputs, max_new_tokens):
        output = inputs.clone()
        for _ in range(max_new_tokens):
            current_seq_length = inputs.size(1)
            if current_seq_length > context_length:
                inputs = inputs[:, -context_length:]
            logits, _ = self(inputs)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat([inputs, idx_next], dim=1)
            output = torch.cat([output, idx_next], dim=1)
        return [decode(out.tolist()) for out in output]

# Re-Initialize the High-Def Model
# Note: We must re-create PositionalEncoding because d_model changed
# The class definition for PositionalEncoding is implicitly used here with new d_model
m = GPT(vocab_size, d_model).to(device)
print(f"High-Def GPT Initialized. d_model: {d_model}")

High-Def GPT Initialized. d_model: 512


In [90]:
# Create new optimizer for the larger model
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
epochs = 2000

print(f"Training High-Def Model (d_model={d_model})...")

for ep in range(epochs):
    xb, yb = train_loader.get_batch()
    
    logits, loss = m(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if ep % 500 == 0 or ep == epochs - 1:
        print(f"Epoch: {ep}\tLoss: {loss.item():.4f}")

print("Training Complete.")

Training High-Def Model (d_model=512)...
Epoch: 0	Loss: 4.5918
Epoch: 500	Loss: 2.4893
Epoch: 1000	Loss: 2.3759
Epoch: 1500	Loss: 2.3583
Epoch: 1999	Loss: 2.3392
Training Complete.


In [92]:
# --- Test Generation ---
print("\n--- Generating Text ---")
start_str = "The "
input_tensor = torch.tensor(encode(start_str), dtype=torch.long, device=device).unsqueeze(0)
generated_text = m.generate(input_tensor, max_new_tokens=100)[0]
print(generated_text)


--- Generating Text ---
The mu\u wef l how Ovin y tousillee
Ane be d I'sit tin I he bet, yoout I re t Ise y p
Ast ldee toum
I yo


In [93]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# --- Configuration ---
n_heads = 4  # We split our 512 dimensions into 4 heads of 128 dims each

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        # Validation: The total model size must be cleanly divisible by the number of heads
        assert (n_heads * self.head_dim == d_model)

        # The Projections
        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        
        # Final aggregation
        self.fc_out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(0.2)

    def forward(self, inputs: torch.Tensor):
        B, seq_length, d_model = inputs.shape
        
        # 1. Project inputs into Q, K, and V
        # Reshape to: (Batch, Seq_Len, n_heads, head_dim) -> Then Swap to: (Batch, n_heads, Seq_Len, head_dim)
        # This separates the "Heads" so they can work in parallel
        Q = self.query(inputs).view(B, seq_length, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = self.key(inputs).view(B, seq_length, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = self.value(inputs).view(B, seq_length, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        
        # 2. Compute attention scores
        # We divide by sqrt(head_dim) to keep gradients stable (Scaled Dot-Product Attention)
        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        # 3. Apply Mask (The Time Shield)
        mask = torch.triu(torch.ones(seq_length, seq_length), diagonal=1).bool().to(inputs.device)
        attention_scores = attention_scores.masked_fill(mask, float('-inf'))
        
        # 4. Softmax & Dropout
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # 5. Aggregate Values
        attention_output = torch.matmul(attention_weights, V)

        # 6. Concatenate heads and restore original shape
        # Swap back: (Batch, Seq_Len, n_heads, head_dim) -> Flatten: (Batch, Seq_Len, d_model)
        attention_output = attention_output.permute(0, 2, 1, 3).contiguous()
        attention_output = attention_output.view(B, seq_length, d_model)

        # 7. Final Linear Transformation
        out = self.fc_out(attention_output)
        
        return out

print("Multi-Head Attention Module Online.")

Multi-Head Attention Module Online.


In [94]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads):
        super().__init__()
        self.wte = nn.Embedding(vocab_size, d_model)
        self.wpe = PositionalEncoding(context_length, d_model)
        
        # NEW: Multi-Head Attention
        self.att = MultiHeadAttention(d_model, n_heads)
        
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        
        self.fcn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        self.dropout = nn.Dropout(0.2)
        self.linear1 = nn.Linear(d_model, vocab_size)

    def forward(self, inputs, targets=None):
        logits = self.wte(inputs)
        logits = self.wpe(logits)
        
        # Attention Block
        att_logits = self.att(logits)
        adn_logits = self.ln1(logits + att_logits)
        
        logits = self.dropout(adn_logits)
        
        # Feed-Forward Block
        fcn_logits = self.fcn(logits)
        logits = self.ln2(logits + fcn_logits)
        
        # Final Projection
        logits = self.linear1(logits)
        
        loss = None
        if targets is not None:
            batch_size, sequence_length, output_dim = logits.shape
            logits = logits.view(batch_size * sequence_length, output_dim)
            targets = targets.view(batch_size * sequence_length)
            loss = F.cross_entropy(logits, targets)
            
        return logits, loss
    
    # (Generate function remains the same)
    def generate(self, inputs, max_new_tokens):
        output = inputs.clone()
        for _ in range(max_new_tokens):
            current_seq_length = inputs.size(1)
            if current_seq_length > context_length:
                inputs = inputs[:, -context_length:]
            logits, _ = self(inputs)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            inputs = torch.cat([inputs, idx_next], dim=1)
            output = torch.cat([output, idx_next], dim=1)
        return [decode(out.tolist()) for out in output]

# Re-Initialize with 4 Heads
m = GPT(vocab_size, d_model, n_heads).to(device)
print(f"GPT Model Upgraded: {n_heads} Attention Heads Active.")

GPT Model Upgraded: 4 Attention Heads Active.


In [95]:
# Create Optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
epochs = 4000

print(f"Training Multi-Head Model ({n_heads} heads)...")

for ep in range(epochs):
    xb, yb = train_loader.get_batch()
    
    logits, loss = m(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if ep % 500 == 0 or ep == epochs - 1:
        print(f"Epoch: {ep}\tLoss: {loss.item():.4f}")

print("Training Complete. The model now sees in multiple dimensions!")

Training Multi-Head Model (4 heads)...
Epoch: 0	Loss: 4.6017
Epoch: 500	Loss: 2.0769
Epoch: 1000	Loss: 1.7576
Epoch: 1500	Loss: 1.6639
Epoch: 2000	Loss: 1.5880
Epoch: 2500	Loss: 1.6333
Epoch: 3000	Loss: 1.6923
Epoch: 3500	Loss: 1.5301
Epoch: 3999	Loss: 1.6549
Training Complete. The model now sees in multiple dimensions!


In [98]:
# --- Test Generation ---
print("\n--- Generating Text ---")
start_str = "Baby Brunson "
input_tensor = torch.tensor(encode(start_str), dtype=torch.long, device=device).unsqueeze(0)
generated_text = m.generate(input_tensor, max_new_tokens=1000)[0]
print(generated_text)


--- Generating Text ---
Baby Brunson and I useWfo\u20 \u2018 take to ting al yaringer anothe a re love, yed I know Apriends this, den arapin my skin
But my night
I must I chumblem lung b, who I led whileft me and baby closing the joy] yesstempteze nvythings
You never bottle,
I will that urre one give way to lling here body the somethings
Maybe you needitcased you, I save my thine
Caustly in this browed
No ountin', been mall then
For ismplaying that tonight it Paslay is hine
My staring ting me

Just black aroundso nsingr, all ong on the trife
Is soblelujacked out my still of nto, "
And if I'm don\"
Onshesreatibled
I rink abough you\u201,
you said fid

Forever
8, her playing utating for wed livings all, liked fold I'll way I'l just lad
She way
If you sturasy wit chy lookin, it on 
I won\u2019t ya you sould now that take all oud, daygrave the body mind
I'm your beeen we'd cleating crs
Darling as thred the proudle on the stading are and The day crowdry made too onlate mode
And it's hampac